# ✅ Solutions — Chapter 6 — Top-down Ontology Development — agentic lab

This is the **solution** notebook: the same lab with all 3 tasks worked. It runs clean end to end, which is what proves the reference implementations satisfy the marking scheme.

> Student version: [`04_agentic_lab.ipynb`](04_agentic_lab.ipynb)

# Chapter 6 — Top-down Ontology Development
### Notebook 4 · Agentic lab — choosing relations, and asking well

*Book reference: Extends §6.1–6.2*

Two things worth doing here. An agent that stops collapsing seven relations into one — and an MDP that **derives DOLCE's decision tree** from a reward function instead of taking it on authority.

In [ ]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

In [ ]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch06_toolkit as ch6
import pandas as pd
logging.getLogger("dspy").setLevel(logging.WARNING)

In [ ]:
import ch06_agentic as AG
from oe_course import evaluation as ev, llm, mdp, optimize as opt
import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

**By the end of this notebook you can:**

1. Build an agent that distinguishes parthood from its impostors.
2. Pair every class in a dataset across the split, and see why an unpaired class is unlearnable.
3. Model **diagnosis** as an MDP with stochastic answers, and read the optimal policy as a decision tree.
4. Show that the derived tree matches the one a foundational ontology ships — and explain what changes it.

> **Prerequisite:** the Chapter 1 agentic lab.

## 1. Tools

`check_chaining` is the one that earns its place: an agent that calls it cannot produce the hand-in-the-orchestra inference, whatever it believes.

In [ ]:
ctx = AG.Ch6Context()
tools = {t.name: t for t in AG.build_toolset(ctx)}
for name, t in tools.items():
    print(f'{name:26s} {list(t.args_schema.model_json_schema().get("properties", {}))}')
    print(f'{"":26s} {t.description.splitlines()[0]}')

In [ ]:
print(tools['classify_relation'].invoke(
    {'part_category': 'amount-of-matter', 'whole_category': 'physical-object'}))
print(tools['check_chaining'].invoke(
    {'first': 'component-of', 'second': 'member-of'}))
print()
for q, a in [('happens', 'false'), ('spatial', 'true'), ('mass', 'true')]:
    print(tools['ask_decision_question'].invoke({'question': q, 'answer': a}))

## 2. The dataset

Sixteen statements, all phrased with the words "part of". The cases are listed as **adjacent pairs of the same relation** because the split alternates — so every relation appears in both halves.

That pairing is not cosmetic. Chapter 5's Exercise 4.1 showed a rule becoming unlearnable when the training data could not exercise it; here the same risk applies to every one of the seven relations at once.

In [ ]:
train, dev = AG.build_dataset('train'), AG.build_dataset('dev')
print(pd.DataFrame([{'id': e.id, 'relation': e.gold_relation,
                     'parthood': e.gold_parthood,
                     'split': 'train' if e in train else 'dev'}
                    for e in AG.build_dataset('all')]).to_string(index=False))

In [ ]:
print('train relations:', sorted({e.gold_relation for e in train}))
print('dev   relations:', sorted({e.gold_relation for e in dev}))
assert {e.gold_relation for e in train} == {e.gold_relation for e in dev}
print('\nEvery relation appears on both sides. Without that, the rules for the\n'
      'dev-only relations could never be learned and the held-out score would\n'
      'be capped for a reason invisible in the report.')

## 3. Baseline: the single-`partOf` modeller

The un-instructed agent does what a great many published ontologies do — answers `component-of` for everything and calls it parthood.

In [ ]:
lm = llm.configure_dspy(AG.PARTWHOLE_RULEBOOK, AG.partwhole_responder)
baseline = AG.PartWholeProgram()
for e in dev[:4]:
    p = baseline(**e.inputs())
    print(f'{e.id:20s} {e.statement}')
    print(f'{"":20s} answered {p.relation} / parthood={p.is_parthood}'
          f'   (gold {e.gold_relation} / {e.gold_parthood})')

In [ ]:
before = ev.evaluate_dataset(baseline, dev, AG.partwhole_scorer)
print('BEFORE:', before['mean_score'])
print('violations:', before['violations'])

In [ ]:
gepa_metric = ev.make_gepa_metric(AG.partwhole_scorer, AG.PARTWHOLE_RULEBOOK)
reflect = llm.reflection_lm(AG.PARTWHOLE_RULEBOOK, AG.partwhole_responder)
tuned = opt.run_gepa(baseline, train, gepa_metric, valset=train,
                     max_metric_calls=100, reflection_lm=reflect)
result = opt.compare(AG.PartWholeProgram(), tuned, dev, AG.partwhole_scorer)
print(result.report())

In [ ]:
found = AG.PARTWHOLE_RULEBOOK.active_in(result.instruction_after)
print(f'rules discovered: {len(found)}/{len(AG.PARTWHOLE_RULEBOOK.ids)}')
for rule_id in sorted(found):
    print('  -', rule_id)
print('missed:', sorted(set(AG.PARTWHOLE_RULEBOOK.ids) - found) or 'none')

> The rule worth noticing is `check-genuine-parthood`. The others fix *which relation* is named; that one fixes whether **anything may be inferred from it**. An agent that gets the name right and parthood wrong will still license the bad chain.

## 4. Deriving the decision tree

Now the interesting MDP. To align a class the agent asks yes/no questions, each costing a little, and commits when more questioning is not worth the price.

| | |
|---|---|
| **S** | the categories still consistent with the answers so far |
| **A** | ask a question that actually splits the set, or commit |
| **T** | **stochastic** — you do not know the answer until you ask |
| **R** | −cost per question; on commit, the probability of being right (`1/k`) |

Committing with `k` candidates left is right with probability `1/k`, so the agent genuinely trades questions against accuracy.

In [ ]:
M = AG.CategoryDiagnosisMDP(question_cost=0.05)
print('reachable candidate sets:', len({s.candidates for s in M.states()}))
print('states (with commit flag):', len(M.states()))
print('\nNote: the full power set of 7 categories would be 128 subsets.\n'
      'Only the ones actually reachable by asking questions are enumerated.')

In [ ]:
V, pi = mdp.value_iteration(M)
s0 = M.initial_state()
print(f'V*(s0) = {V[s0]:.4f}')
print(f'  (1.0 accuracy minus the expected cost of the questions asked)')

### The optimal policy, rendered as the tree it is:

In [ ]:
for line in M.decision_tree(pi):
    print(line)

> **Compare that with §6.1.** The optimiser split on `happens?` first — the endurant/perdurant distinction, which is exactly where every foundational ontology starts. Nobody told it that; it followed from the question being the one that best halves the candidate set.

This is the most satisfying result in the course: a decision tree that textbooks present as received wisdom, **derived** from a cost model.

In [ ]:
import random
random.seed(0)
lengths = []
for _ in range(400):
    ep = mdp.run_episode(M, mdp.greedy_policy(pi))
    lengths.append(len(ep) - 1)      # questions asked before committing
print(f'average questions asked: {sum(lengths)/len(lengths):.2f}')
print(f'range: {min(lengths)}-{max(lengths)}')
print('\nA perdurant is settled in two questions; an endurant needs three or\n'
      'four. The tree is unbalanced because the categories are.')

### Task 4.1 — Make questions expensive

Raise `question_cost` until the optimal policy stops asking altogether. Report the threshold and explain it in terms of the accuracy being bought.

In [ ]:
rows = []
for cost in [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]:
    Mc = AG.CategoryDiagnosisMDP(question_cost=cost)
    Vc, pic = mdp.value_iteration(Mc)
    tree = Mc.decision_tree(pic)
    asks = sum(1 for line in tree if '?' in line)
    rows.append({'question_cost': cost,
                 'V*': round(Vc[Mc.initial_state()], 4),
                 'questions in tree': asks})
print(pd.DataFrame(rows).to_string(index=False))
silent = [r for r in rows if r['questions in tree'] == 0]
print(f"\nthe agent stops asking at cost >= {silent[0]['question_cost'] if silent else 'never in this range'}")
print('Committing blind to one of seven categories is worth 1/7 = 0.143. A\n'
      'question is worth asking only while it buys more accuracy than it costs,\n'
      'so once questions get expensive enough the optimal ontologist guesses --\n'
      'which is a statement about budgets, not about rigour.')

**Checks for Task 4.1.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert [r['question_cost'] for r in rows] == [0.0, 0.05, 0.1, 0.2, 0.3, 0.5]
assert rows[0]['questions in tree'] > 0, 'free questions should always be asked'
# Raising the cost of a question cannot raise the optimal value, nor make the
# agent ask more of them.
assert rows[-1]['V*'] <= rows[0]['V*'] + 1e-9
assert rows[-1]['questions in tree'] <= rows[0]['questions in tree']

### Task 4.2 — Remove a question and watch the tree adapt

Drop `happens` from the question set and re-derive the tree. Report what it costs in expected value, and what the new first question is.

In [ ]:
reduced = tuple(q for q in ch6.DECISION_QUESTIONS if q != 'happens')
M2 = AG.CategoryDiagnosisMDP(question_cost=0.05, questions=reduced)
V2, pi2 = mdp.value_iteration(M2)
print('remaining questions:', reduced)
print(f'V* with all five  : {V[s0]:.4f}')
print(f'V* without happens: {V2[M2.initial_state()]:.4f}')
print('\nnew tree:')
for line in M2.decision_tree(pi2):
    print(line)
print('\nThe endurant/perdurant question is the most informative single question\n'
      'available, so removing it costs value -- but the optimiser simply\n'
      're-plans around the loss rather than failing. That is the practical\n'
      'argument for deriving a decision procedure instead of hard-coding one:\n'
      'when the available evidence changes, the procedure should change with it.')

**Checks for Task 4.2.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert V2[M2.initial_state()] <= V[s0]

### Task 4.3 — Give the agent the chaining tool and prove it cannot be fooled

Show that an agent using `check_chaining` refuses the hand/orchestra inference, and that the refusal is grounded in the relation properties rather than in the prompt.

In [ ]:
ctx2 = AG.Ch6Context()
t2 = {t.name: t for t in AG.build_toolset(ctx2)}

hand = json.loads(t2['classify_relation'].invoke(
    {'part_category': 'physical-object', 'whole_category': 'physical-object'}))
musician = json.loads(t2['classify_relation'].invoke(
    {'part_category': 'physical-object', 'whole_category': 'collection'}))
print('hand -> musician  :', hand['relation'], '(parthood', hand['parthood'], ')')
print('musician -> orch. :', musician['relation'], '(parthood', musician['parthood'], ')')

chain = json.loads(t2['check_chaining'].invoke(
    {'first': hand['relation'], 'second': musician['relation']}))
print('\nchain valid?', chain['valid'])
print('reason      :', chain['reason'])
print('\ntool calls:', ctx2.log.names())
print('\nThe refusal comes from the relation table, not from an instruction\n'
      'telling the agent that orchestras are special. Encoding a distinction in\n'
      'a TOOL rather than a PROMPT is what makes it survive prompt optimisation,\n'
      'model swaps, and the next engineer.')

**Checks for Task 4.3.** These assertions are the marking scheme. They run in both variants — here they pass once your implementation is right.

In [ ]:
assert not chain['valid']

## Chapter 6 in the course arc

| | Ch. 3 | Ch. 4 | Ch. 5 | Ch. 6 |
|---|---|---|---|---|
| MDP | budgeted, stochastic | construction | plan under prerequisites | **diagnosis (derives a decision tree)** |
| grader | free oracle | profile table | measured CQ coverage | relation taxonomy |
| the error it prevents | unsound entailment | profile violation | ontologically wrong axiom | **unsound part-whole chaining** |

Chapters 5 and 6 together make one argument. Chapter 5 found an error a reasoner could not see; Chapter 6 supplied the vocabulary to repair it. Neither was a matter of more logic — both were a matter of **making more distinctions**.

That is the top-down half of ontology development. Chapter 7 goes the other way: extracting an ontology from text and data, where you get no distinctions for free at all.